# Layer 1: Pattern Detection
This layer detects fake news using pattern-based analysis.

## Part 1: Training

### 1. Setup and Imports
Importing necessary libraries and defining helper functions.

In [1]:
import os
import re
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

def extract_stylometric_features(texts):
    features = pd.DataFrame()
    features['text_len'] = texts.apply(lambda x: len(str(x)))
    features['word_count'] = texts.apply(lambda x: len(str(x).split()))
    features['avg_word_len'] = features['text_len'] / (features['word_count'] + 1)
    features['exclamation_count'] = texts.apply(lambda x: str(x).count('!'))
    features['question_count'] = texts.apply(lambda x: str(x).count('?'))
    features['caps_ratio'] = texts.apply(lambda x: sum(1 for word in str(x).split() if word.isupper()) / (len(str(x).split()) + 1))
    return features.values

### 2. Loading and Preprocessing Data
Loading datasets, assigning labels, and cleaning text (removing Reuters tags).

In [2]:
print("Loading data...")
DIR = os.getcwd()
try:
    fake_df = pd.read_csv(os.path.join(DIR, 'Fake.csv'))
    true_df = pd.read_csv(os.path.join(DIR, 'True.csv'))
except FileNotFoundError:
    print("Warning: Fake.csv or True.csv not found!")
    fake_df = pd.DataFrame({'text': [], 'label': []})
    true_df = pd.DataFrame({'text': [], 'label': []})

fake_df['label'] = 1
true_df['label'] = 0

df = pd.concat([fake_df, true_df], ignore_index=True)
if not df.empty:
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print("Cleaning text...")
    df['text'] = df['text'].apply(lambda x: re.sub(r'^.*?\(Reuters\) - ', '', str(x)))

    X = df['text']
    y = df['label']
else:
    X, y = [], []

Loading data...


Cleaning text...


### 3. Feature Extraction (TF-IDF)
Vectorizing text with N-Grams (1 to 3 words) and saving the vectorizer.

In [3]:
print("Vectorizing text with N-Grams (1 to 3 words)...")
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    tfidf = TfidfVectorizer(stop_words='english', max_df=0.7, ngram_range=(1, 3), max_features=15000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)
    
    joblib.dump(tfidf, os.path.join(DIR, 'tfidf_vectorizer.pkl'))

Vectorizing text with N-Grams (1 to 3 words)...


### 4. Feature Extraction (Stylometric)
Extracting stylometric features, scaling them, and combining with TF-IDF features.

In [4]:
print("Extracting stylometric features...")
if len(X) > 0:
    X_train_stylo = extract_stylometric_features(X_train)
    X_test_stylo = extract_stylometric_features(X_test)
    
    scaler = StandardScaler()
    X_train_stylo_scaled = scaler.fit_transform(X_train_stylo)
    X_test_stylo_scaled = scaler.transform(X_test_stylo)
    
    joblib.dump(scaler, os.path.join(DIR, 'scaler.pkl'))
    
    X_train_combined = hstack([X_train_tfidf, X_train_stylo_scaled])
    X_test_combined = hstack([X_test_tfidf, X_test_stylo_scaled])

Extracting stylometric features...


### 5. Model Training & Evaluation
Training models (Logistic Regression, Passive Aggressive, Random Forest, XGBoost) and identifying the best one based on F1-Score.

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Passive Aggressive': PassiveAggressiveClassifier(max_iter=50),
    'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

best_f1 = 0
best_model = None
best_model_name = ""

print("Training models...")
if len(X) > 0:
    for name, model in models.items():
        model.fit(X_train_combined, y_train)
        preds = model.predict(X_test_combined)
        f1 = f1_score(y_test, preds)
        print(f"[{name}] F1-Score: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            best_model = model
            best_model_name = name

    print(f"\nBest Model: {best_model_name} (F1: {best_f1:.4f})")
    joblib.dump(best_model, os.path.join(DIR, 'best_model.pkl'))
    print("Saved 'best_model.pkl', 'tfidf_vectorizer.pkl', and 'scaler.pkl'")

Training models...


[Logistic Regression] F1-Score: 0.9821


[Passive Aggressive] F1-Score: 0.9888


[Random Forest] F1-Score: 0.9876


[XGBoost] F1-Score: 0.9906

Best Model: XGBoost (F1: 0.9906)
Saved 'best_model.pkl', 'tfidf_vectorizer.pkl', and 'scaler.pkl'


## Part 2: Prediction

### 6. Loading Artifacts
Loading the trained model and preparing for predictions on new data.

In [6]:
DIR = os.getcwd()

try:
    vectorizer = joblib.load(os.path.join(DIR, 'tfidf_vectorizer.pkl'))
    model = joblib.load(os.path.join(DIR, 'best_model.pkl'))
    scaler = joblib.load(os.path.join(DIR, 'scaler.pkl'))
    print("Artifacts loaded successfully!")
except FileNotFoundError:
    vectorizer = None
    model = None
    scaler = None
    print("Warning: Model files not found. Please run the training cells first.")

Artifacts loaded successfully!


### 7. Prediction Logic
Defining the prediction function which processes new texts using identical feature extraction pipelines.

In [7]:
def predict_layer1(article_text: str) -> dict:
    if not model or not vectorizer or not scaler:
        return {"label": "ERROR", "confidence": 0.0, "reason": "Model not trained yet."}
    
    text_tfidf = vectorizer.transform([article_text])
    
    text_series = pd.Series([article_text])
    stylo_features = extract_stylometric_features(text_series)
    stylo_scaled = scaler.transform(stylo_features)
    
    text_combined = hstack([text_tfidf, stylo_scaled])
    
    prediction = model.predict(text_combined)[0]
    
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(text_combined)[0]
        confidence = float(max(probabilities))
    elif hasattr(model, "decision_function"):
        dec_val = abs(model.decision_function(text_combined)[0])
        confidence = float(min(0.5 + (dec_val / 2.0), 0.99))
    else:
        confidence = 0.85
        
    label_str = "FAKE" if prediction == 1 else "REAL"
    reason = "Lexical patterns and N-Grams strongly match known text characteristics"
    
    return {
        "label": label_str,
        "confidence": round(confidence, 4),
        "reason": f"{reason} for {label_str} news."
    }

# Quick local test
sample_text = "Scientists have just discovered a massive planet entirely made of chocolate!"
print(predict_layer1(sample_text))

{'label': 'FAKE', 'confidence': 0.992, 'reason': 'Lexical patterns and N-Grams strongly match known text characteristics for FAKE news.'}
